# Multimer analysis III - scANVI intergration
scANVI offers label transfer between a main dataset (or atlas) and a current unlabelled dataset in question. We can use this to predict the cell identity of multimer-sorted cells labelling them with T-cell phenotypes relevant for our study

#### This was performed at the high-performance PC

In [ ]:
import anndata as ad
import scanpy as sc
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import muon as mu
import scvi
import torch
from mudata import MuData
# from scipy.stats import binomtest
# from scipy.stats import mannwhitneyu
# import string

# Helps to define plotting area
import matplotlib.pyplot as plt
# from matplotlib import rcParams 
# from matplotlib.pyplot import rc_context
# import matplotlib.gridspec as gridspec
# import matplotlib.colors as mcolors
# from matplotlib import colors
# # from matplotlib_venn import venn2
# from scipy.stats import spearmanr
# import matplotlib.patches as mpatches
# from pathlib import Path

## For making text editable in .pdfs once exported use cairo backend instead of agg
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# set seed
np.random.seed(44)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999

In [ ]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

In [ ]:
torch.set_float32_matmul_precision("high")

# Check GPU support

In [ ]:
# Check PyTorch sees your GPU
print(torch.cuda.is_available())       # Should print: True
print(torch.cuda.get_device_name(0))   # e.g. "NVIDIA GeForce RTX 3080"

# Check scvi-tools version
print(scvi.__version__)

# Loading data
## Step 1 - Load all multimer data

In [ ]:
mdata = mu.read(filename=r"NK13_PostIV_scVI_intergration_and_annotation.h5mu")
len(mdata)

## Step 2 - Load the main CD8 data from sorted cells (sdata)

In [ ]:
sdata = mu.read(filename=r"2025_12_CD8_annotated.h5mu")
len(sdata)

# Combine the datasets

In [ ]:
combined = MuData({
    "gex":  ad.concat([sdata["gex"],  mdata["gex"]],  label="batch", keys=["annotated", "unannotated"], axis=0, join="outer"),
    "airr": ad.concat([sdata["airr"], mdata["airr"]], label="batch", keys=["annotated", "unannotated"], axis=0, join="outer"),
    "prot": ad.concat([sdata["prot"], mdata["prot"]],  label="batch", keys=["annotated", "unannotated"], axis=0, join="outer"),
})

In [ ]:
combined["gex"].obs.columns.tolist()

In [ ]:
### Clean up the .obs data frame
cols_to_keep = ["batch", "donor", "Compartment", "clone_handle", "type_x_clone_count", "total_clone_count", "final_ann", "Differentiation", "L3_ann_overview", "L3_ann",
                'IR_VJ_1_v_call',  'IR_VJ_1_j_call', 'IR_VJ_1_junction_aa', 'IR_VDJ_1_v_call',  'IR_VDJ_1_d_call',  'IR_VDJ_1_j_call', 'IR_VDJ_1_junction_aa', 'HLA',
                'handle', 'Peptide', 'Sequence', "Virus", "Antigen", 'total_agspecific_cells', 'CSF-biased score', 'PB-biased score']  # whatever columns you need

combined["gex"].obs = combined["gex"].obs.loc[:, cols_to_keep]

# scANVI intergration
- See: https://docs.scvi-tools.org/en/stable/tutorials/notebooks/scrna/tabula_muris.html
- See: https://docs.scvi-tools.org/en/1.4.0/tutorials/notebooks/scrna/seed_labeling.html

We assume that the SS2 data is annotated and the 10X data is not. Only the labels of the cells from the labelled SS2 dataset will be kept in the adata.obs column 'celltype_scanvi'. All the 10X cells will have 'celltype_scanvi' of value ‘Unknown’.

### Steps:
- Work with a combined dataset
- Annotate all NA values as "Unknown" among L3_ann
- Define highly variable genes. To speed up scANVI, we will use a lower number of highly variable genes
- Define the model parameters

In [ ]:
# Specifically name the unlabelled cells Unknown
combined["gex"].obs.L3_ann = pd.Categorical(np.where(combined["gex"].obs.L3_ann.isna(), "Unknown", combined["gex"].obs.L3_ann))

In [ ]:
# scVI works better with fewer genes - subset to the highly variable genes
sc.pp.highly_variable_genes(
    combined["gex"],
    flavor="seurat_v3",
    n_top_genes=2000,
    layer="counts",
    batch_key="batch",
    subset=True,
)

In [ ]:
# Set up the model (scANVI requires a 'labels_key' with cell type annotations)
scvi.model.SCVI.setup_anndata(combined["gex"], batch_key="batch", labels_key="L3_ann")

In [ ]:
scvi_model = scvi.model.SCVI(combined["gex"], n_latent=30, n_layers=2)

In [ ]:
scvi_model.train(100)

In [ ]:
scanvi_model = scvi.model.SCANVI.from_scvi_model(scvi_model, "Unknown")

In [ ]:
scanvi_model.train(25)

In [ ]:
SCANVI_LATENT_KEY = "X_scANVI"
SCANVI_PREDICTIONS_KEY = "C_scANVI"

combined["gex"].obsm[SCANVI_LATENT_KEY] = scanvi_model.get_latent_representation(combined["gex"])
combined["gex"].obs[SCANVI_PREDICTIONS_KEY] = scanvi_model.predict(combined["gex"])

In [ ]:
sc.pp.neighbors(combined["gex"], use_rep=SCANVI_LATENT_KEY)
sc.tl.umap(combined["gex"])

In [ ]:
combined["gex"]

In [ ]:
sc.pl.umap(combined["gex"], color=["L3_ann", "C_scANVI", "Compartment", "_scvi_labels", "L3_ann_overview", "HLA", "Virus"], )

# Export 

In [ ]:
mdata.write(filename="NK13_PostV_scANVI_labeltransfer_intergration.h5mu")